In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

#### Load the data

In [2]:
# Convert images to tensors
transform = transforms.ToTensor()

# Download and load the MNIST training dataset
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Create the DataLoader
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True
)

print(f"Number of training samples: {len(train_dataset)}")

Number of training samples: 60000


#### Building the neural network

In [3]:
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),          # Convert 28x28 image to a vector of 784 values
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)     # 10 output classes (digits 0–9)
        )

    def forward(self, x):
        return self.network(x)


model = SimpleNN()

print(model)

SimpleNN(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


#### Loss function and Optimiser

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.01
)

#### Train the model

In [12]:
num_epochs = 5

for epoch in range(num_epochs):

    print(f"\nEpoch {epoch + 1}")

    for images, labels in train_loader:

        # ------------------------------------------------
        # 1. Select a batch of data
        # ------------------------------------------------
        # The DataLoader provides one mini-batch of images and labels.

        # Clear gradients from the previous iteration
        optimizer.zero_grad()

        # ------------------------------------------------
        # 2. Forward pass
        # ------------------------------------------------
        outputs = model(images)

        # Compute the loss
        loss = criterion(outputs, labels)

        # ------------------------------------------------
        # 3. Backward pass
        # ------------------------------------------------
        # Compute gradients for every trainable parameter
        loss.backward()

        # ------------------------------------------------
        # 4. Update the weights
        # ------------------------------------------------
        # Use the gradients to update the model parameters
        optimizer.step()

    print(f"Loss: {loss.item():.4f}")


Epoch 1
Loss: 0.2992

Epoch 2
Loss: 0.2750

Epoch 3
Loss: 0.3232

Epoch 4
Loss: 0.4034

Epoch 5
Loss: 0.1402


#### Test model in one batch

In [13]:
model.eval()

with torch.no_grad():

    images, labels = next(iter(train_loader))

    outputs = model(images)

    predictions = torch.argmax(outputs, dim=1)

print("Predicted labels:")
print(predictions[:10])

print("\nActual labels:")
print(labels[:10])

Predicted labels:
tensor([8, 7, 4, 1, 7, 5, 2, 9, 6, 2])

Actual labels:
tensor([8, 7, 4, 5, 3, 5, 2, 9, 6, 2])
